## Trade have requested themes using attributes that currently don't feature in TargetingAttribute Options

#### Check feature exists in product_catalog_history by looking at existing columns

In [0]:
%sql
describe warehouse.product_catalog_history

#### Verify the values they are filtering for exist in column by filtering for values and sense checking outputs

In [0]:
%sql

select distinct pid, title from warehouse.product_catalog_history
where collaboration  = 'minecraft'

#### run parse_attributes with new attribute added to config and check output table contains new attribute

In [0]:
%sql
select * from ds_sandbox.next_uk_nextads_attribute_set where attribute = 'collaboration'

#### check values attached to new attribute

In [0]:
%sql
select attribute, value from ds_sandbox.next_uk_nextads_attribute_set
qualify max(rundate) over() = rundate

## Trade have requested new themes

#### Add all new themes to Theme Mapping in dev control sheet and run parse_theme_mapping setting REFRESH_THEME_DATE to True. Then check new themes are present in table

In [0]:
%sql

select * from ds_sandbox.next_uk_nextads_item_themes
where theme in ('kids minecraft',
'kids spiderman',
'beauty korean skincare',
'start rite',
'kickers',
'hype',
'clarks',
'baker by ted baker')

#### Check distribution of top mapped themes, are any of the new themes mapped to a too small number of items?


In [0]:
%sql

select distinct theme, count(*) as count
from ds_sandbox.next_uk_nextads_item_themes
where theme_rank = 1
group by all
order by count desc

#### Run build_markov_chain with REFRESH_MODEL_DATE set to True and check distribution of how many customers are assigned new themes

In [0]:
%sql
with t0 as (
  select *, rank() over (partition by accountnumber order by probaggrebased desc) as rank
  from ds_sandbox.next_uk_nextads_next_theme_scores_latest
)
, t1 as(
select distinct accountnumber, nexttheme from t0 
where nexttheme in ('kids minecraft',
'kids spiderman',
'beauty korean skincare',
'start rite',
'kickers',
'hype',
'clarks',
'baker by ted baker') and rank <=3
)

select distinct nexttheme, count(*) as count from t1 
group by all
order by count desc

#### Check overall distribution of top assigned themes for sandbox and warehouse tables to confirm nothing wild/ unexplainable has happened

In [0]:
%sql
with t0 as (
  select *, rank() over (partition by accountnumber order by probaggrebased desc) as rank
  from ds_sandbox.next_uk_nextads_next_theme_scores_latest
)
select distinct nexttheme, count(*) as count 
from t0 
where rank = 1
group by all
order by count desc

In [0]:
%sql
with t0 as (
  select *, rank() over (partition by accountnumber order by probaggrebased desc) as rank
  from warehouse.next_uk_nextads_next_theme_scores_latest
)
select distinct nexttheme, count(*) as count 
from t0 
where rank = 1
group by all
order by count desc

## Once all above checks have been done, new themes (and attributes) can be pasted over to the prod control sheet